# Stage 2 Fine-Tune

Fine-tune v31 (solo pacing model) with opponents.

**Why:** v31 learned genuine pacing in solo time trials (Q1=-0.16, Q2=0.00, Q3=0.26, Q4=0.75, finishing at 12.2 m/s with 24% stamina). But it finishes 2nd-3rd against opponents because it brakes/coasts while they push. Stage 2 teaches it to compete for placement without losing the pacing foundation.

**Strategy:**
- Load v31 best_model (pacing locked in)
- Add opponents (horse_count=4, scripted strategies, randomized attributes)
- Gentle LR (5e-5 → 5e-6) to adapt without destroying pacing
- 1M timesteps
- Optional self-play from v31 checkpoint

**Base model:** `hr-checkpoints-v2/v31/best_model.zip`

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

VERSION = "v32"
BASE_VERSION = "v31"  # solo pacing model

TRACK = "tracks/test_oval.json"
HORSE_COUNT = 4          # stage 2: add opponents
MAX_STEPS = 5000
N_ENVS = 12

TOTAL_TIMESTEPS = 1_000_000

# Fine-tune LR: gentler than full training to preserve pacing
LR_START = 5e-5
LR_END = 5e-6

# Self-play: use v31 as opponent so agent races against its own pacing strategy
SELF_PLAY_MODEL = None       # set to f"{DRIVE_BASE}/{BASE_VERSION}/best_model.zip" to enable
SELF_PLAY_RATIO = 0.5        # fraction of opponents that use self-play

# Google Drive
DRIVE_BASE = "/content/drive/MyDrive/hr-checkpoints-v2"
DRIVE_DIR = f"{DRIVE_BASE}/{VERSION}"
BASE_MODEL = f"{DRIVE_BASE}/{BASE_VERSION}/best_model.zip"

RESUME = 'auto'       # 'auto', False, or path to .zip
LOG_EVERY = 10
SAVE_EVERY = 50_000

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = "/content/hr-simulation"
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/ue-too/hr-simulation.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")
!git log --oneline -3

In [ ]:
!pip install -q uv
!uv pip install --system -e "."

## Resolve Checkpoint

In [ ]:
import glob
from pathlib import Path

os.makedirs(DRIVE_DIR, exist_ok=True)

restore_path = None
restored_timesteps = 0

if RESUME is False:
    # Fresh fine-tune from base model
    restore_path = BASE_MODEL
    print(f"Fine-tuning from base: {restore_path}")
elif RESUME == 'auto':
    # Check for existing v28 checkpoints first
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, "checkpoint_*.zip")), key=os.path.getmtime)
    if zips:
        restore_path = zips[-1]
        stem = Path(restore_path).stem
        parts = stem.split("_")
        for i, p in enumerate(parts):
            if p == "steps" and i > 0:
                try:
                    restored_timesteps = int(parts[i - 1])
                except ValueError:
                    pass
        print(f"Auto-resume: {restore_path}")
        print(f"  Restored timesteps: {restored_timesteps:,}")
    else:
        final = os.path.join(DRIVE_DIR, "final_model.zip")
        if os.path.exists(final):
            restore_path = final
            restored_timesteps = TOTAL_TIMESTEPS
            print(f"Found final model — training already complete!")
        else:
            # No v28 checkpoint — start from base model
            restore_path = BASE_MODEL
            print(f"No {VERSION} checkpoint — fine-tuning from base: {restore_path}")
elif isinstance(RESUME, str):
    restore_path = RESUME
    print(f"Resuming from: {restore_path}")

remaining_timesteps = max(0, TOTAL_TIMESTEPS - restored_timesteps)
print(f"\nVersion: {VERSION} (fine-tune from {BASE_VERSION})")
print(f"Track: {TRACK} | Horses: {HORSE_COUNT}")
print(f"Total: {TOTAL_TIMESTEPS:,} | Remaining: {remaining_timesteps:,}")
print(f"Checkpoint dir: {DRIVE_DIR}")

## Training

In [ ]:
import time
import json
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv

from horse_racing.env import HorseRacingSingleEnv


class ColabLoggingCallback(BaseCallback):
    """Logs episode stats, saves history, and preserves the best model."""

    def __init__(self, log_every=10, total_timesteps=1_000_000,
                 history_path=None, best_model_path=None, best_window=20):
        super().__init__(verbose=0)
        self._log_every = log_every
        self._total_timesteps = total_timesteps
        self._history_path = history_path
        self._best_model_path = best_model_path
        self._best_window = best_window
        self._best_avg = -float("inf")
        self._ep_count = 0
        self._start = time.time()
        self.history = []

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self._ep_count += 1
                ep = info["episode"]
                self.history.append({
                    "episode": self._ep_count,
                    "timesteps": self.num_timesteps,
                    "reward": float(ep["r"]),
                    "length": int(ep["l"]),
                })
                if self._ep_count % self._log_every == 0:
                    elapsed = time.time() - self._start
                    ts = self.num_timesteps
                    rate = ts / elapsed if elapsed > 0 else 0
                    pct = ts / self._total_timesteps * 100
                    eta_s = (self._total_timesteps - ts) / rate if rate > 0 else 0
                    eta_m = eta_s / 60
                    recent = self.history[-10:]
                    avg_r = np.mean([h["reward"] for h in recent])
                    avg_l = np.mean([h["length"] for h in recent])
                    print(
                        f"  [{pct:5.1f}%] ep {self._ep_count:5d} | "
                        f"reward: {avg_r:8.3f} | "
                        f"len: {avg_l:6.0f} | "
                        f"ts: {ts:>10,} | "
                        f"{rate:,.0f} ts/s | "
                        f"ETA: {eta_m:.0f}m"
                    )

                # Save best model when rolling average improves
                if (self._best_model_path
                        and self._ep_count >= self._best_window):
                    window = [h["reward"] for h in self.history[-self._best_window:]]
                    avg = np.mean(window)
                    if avg > self._best_avg:
                        self._best_avg = avg
                        self.model.save(self._best_model_path)
                        print(f"  ** New best model saved (avg {avg:.2f}, ep {self._ep_count}) **")

                # Save history periodically
                if self._history_path and self._ep_count % (self._log_every * 5) == 0:
                    with open(self._history_path, "w") as f:
                        json.dump(self.history, f)
        return True


# Load self-play predict function if configured
_self_play_predict_fn = None
if SELF_PLAY_MODEL and os.path.exists(SELF_PLAY_MODEL):
    from horse_racing.opponents.self_play import make_self_play_predict
    _self_play_predict_fn = make_self_play_predict(SELF_PLAY_MODEL)
    print(f"Self-play opponent loaded: {SELF_PLAY_MODEL}")
    print(f"Self-play ratio: {SELF_PLAY_RATIO:.0%} of opponents")
elif SELF_PLAY_MODEL:
    print(f"WARNING: Self-play model not found: {SELF_PLAY_MODEL}")
else:
    print("Self-play: disabled (scripted opponents only)")


def make_env(track_path, horse_count, max_steps, self_play_fn=None, self_play_ratio=0.0):
    def _init():
        return Monitor(HorseRacingSingleEnv(
            track_path=track_path, horse_count=horse_count, max_steps=max_steps,
            self_play_predict_fn=self_play_fn,
            self_play_ratio=self_play_ratio,
        ))
    return _init

In [ ]:
if remaining_timesteps <= 0:
    print("Training already complete — skip to evaluation or export.")
else:
    vec_env = SubprocVecEnv([
        make_env(TRACK, HORSE_COUNT, MAX_STEPS, _self_play_predict_fn, SELF_PLAY_RATIO)
        for _ in range(N_ENVS)
    ])

    def linear_schedule(progress_remaining: float) -> float:
        return LR_END + (LR_START - LR_END) * progress_remaining

    assert restore_path, "Fine-tune requires a base model"
    print(f"Loading model from {restore_path}")
    model = PPO.load(restore_path, env=vec_env, device="cpu")
    model.learning_rate = linear_schedule

    print(f"Params: {sum(p.numel() for p in model.policy.parameters()):,}")
    print(f"LR schedule: {LR_START} → {LR_END} (fine-tune, gentler than full training)")
    if _self_play_predict_fn:
        print(f"Self-play: {SELF_PLAY_RATIO:.0%} of opponents from {SELF_PLAY_MODEL}")
    print()

In [ ]:
if remaining_timesteps <= 0:
    print("Skipping training.")
else:
    history_path = os.path.join(DRIVE_DIR, "history.json")
    best_model_path = os.path.join(DRIVE_DIR, "best_model")

    checkpoint_cb = CheckpointCallback(
        save_freq=max(SAVE_EVERY // N_ENVS, 1),
        save_path=DRIVE_DIR,
        name_prefix="checkpoint",
    )
    logging_cb = ColabLoggingCallback(
        log_every=LOG_EVERY,
        total_timesteps=remaining_timesteps,
        history_path=history_path,
        best_model_path=best_model_path,
        best_window=50,
    )

    print(f"Fine-tuning: {remaining_timesteps:,} timesteps")
    print(f"Track: {TRACK} | Horses: {HORSE_COUNT} | Envs: {N_ENVS}")
    print(f"Checkpoints -> {DRIVE_DIR}")
    print(f"Best model  -> {best_model_path}.zip")
    print()

    start_time = time.time()
    try:
        model.learn(
            total_timesteps=remaining_timesteps,
            callback=[checkpoint_cb, logging_cb],
            reset_num_timesteps=True,
        )
    except KeyboardInterrupt:
        print("\nTraining interrupted.")

    final_path = os.path.join(DRIVE_DIR, "final_model")
    model.save(final_path)
    with open(history_path, "w") as f:
        json.dump(logging_cb.history, f)

    elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"Done — {elapsed/60:.1f} min, {logging_cb._ep_count} episodes")
    print(f"Final model: {final_path}.zip")
    if logging_cb._best_avg > -float("inf"):
        print(f"Best model:  {best_model_path}.zip (avg {logging_cb._best_avg:.2f})")
    print(f"{'='*60}")
    vec_env.close()

## Training Curves

In [ ]:
import matplotlib.pyplot as plt

history_path = os.path.join(DRIVE_DIR, "history.json")
if 'logging_cb' in dir() and logging_cb.history:
    history = logging_cb.history
elif os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
else:
    history = []
    print("No history found.")

if history:
    rewards = [h["reward"] for h in history]
    lengths = [h["length"] for h in history]

    window = min(50, len(rewards) // 4) or 1
    smooth_r = np.convolve(rewards, np.ones(window)/window, mode='valid')
    smooth_l = np.convolve(lengths, np.ones(window)/window, mode='valid')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(rewards, alpha=0.2, color='blue')
    ax1.plot(range(window-1, len(rewards)), smooth_r, color='blue', linewidth=2)
    ax1.set_xlabel('Episode'); ax1.set_ylabel('Reward'); ax1.set_title('Episode Reward')
    ax1.grid(True, alpha=0.3)
    ax2.plot(lengths, alpha=0.2, color='green')
    ax2.plot(range(window-1, len(lengths)), smooth_l, color='green', linewidth=2)
    ax2.set_xlabel('Episode'); ax2.set_ylabel('Length (ticks)'); ax2.set_title('Episode Length')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Episodes: {len(rewards)}")
    print(f"Last 10 avg reward: {np.mean(rewards[-10:]):.3f}")
    print(f"Last 10 avg length: {np.mean(lengths[-10:]):.0f}")

## Evaluation

In [ ]:
from horse_racing.action import decode_action

# Prefer best_model (peak performance) over final_model
best_path = os.path.join(DRIVE_DIR, "best_model.zip")
final_path = os.path.join(DRIVE_DIR, "final_model.zip")

if os.path.exists(best_path):
    eval_path = best_path
elif os.path.exists(final_path):
    eval_path = final_path
else:
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, "checkpoint_*.zip")), key=os.path.getmtime)
    if zips:
        eval_path = zips[-1]
    else:
        raise FileNotFoundError("No model found to evaluate")

eval_model = PPO.load(eval_path, device="cpu")
print(f"Evaluating: {eval_path}\n")

env = HorseRacingSingleEnv(track_path=TRACK, horse_count=HORSE_COUNT, max_steps=MAX_STEPS)
for ep in range(10):
    obs, info = env.reset()
    total_r = 0.0
    steps = 0
    tang_history = []
    while True:
        action, _ = eval_model.predict(obs, deterministic=True)
        t, n = decode_action(int(action))
        tang_history.append(t)
        obs, r, term, trunc, info = env.step(int(action))
        total_r += r
        steps += 1
        if term or trunc:
            break
    status = f"#{info['finish_order']}" if info['finish_order'] else "DNF"
    n_s = len(tang_history)
    q = n_s // 4
    q_avgs = [np.mean(tang_history[i*q:(i+1)*q if i<3 else n_s]) for i in range(4)]
    print(f"  ep {ep+1:2d}: {status} r={total_r:7.2f} stam={info['stamina']:5.1f} | Q1={q_avgs[0]:.2f} Q2={q_avgs[1]:.2f} Q3={q_avgs[2]:.2f} Q4={q_avgs[3]:.2f}")
env.close()

## Export to ONNX

In [ ]:
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
from horse_racing.core.observation import OBS_SIZE
from horse_racing.action import NUM_ACTIONS


class PolicyWrapper(nn.Module):
    def __init__(self, policy):
        super().__init__()
        self.features_extractor = policy.features_extractor
        self.mlp_extractor = policy.mlp_extractor
        self.action_net = policy.action_net

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        features = self.features_extractor(obs)
        latent_pi, _ = self.mlp_extractor(features)
        return self.action_net(latent_pi)


# Prefer best_model (peak performance) over final_model
best_path = os.path.join(DRIVE_DIR, "best_model.zip")
final_path = os.path.join(DRIVE_DIR, "final_model.zip")

if os.path.exists(best_path):
    export_path = best_path
elif os.path.exists(final_path):
    export_path = final_path
else:
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, "checkpoint_*.zip")), key=os.path.getmtime)
    export_path = zips[-1] if zips else None
if not export_path:
    raise FileNotFoundError("No model found to export")

export_model = PPO.load(export_path, device="cpu")
print(f"Exporting from: {export_path}")
wrapper = PolicyWrapper(export_model.policy).cpu()
wrapper.eval()

dummy = torch.zeros(1, OBS_SIZE, dtype=torch.float32)
onnx_path = os.path.join(DRIVE_DIR, f"{VERSION}_jockey.onnx")

torch.onnx.export(
    wrapper, dummy, onnx_path,
    input_names=["obs"], output_names=["actions"],
    dynamic_axes={"obs": {0: "batch"}, "actions": {0: "batch"}},
    opset_version=17,
)

# Ensure weights are embedded inline (not external .data file)
onnx_model = onnx.load(onnx_path)
has_external = any(
    t.HasField("data_location") and t.data_location == onnx.TensorProto.EXTERNAL
    for t in onnx_model.graph.initializer
)
if has_external:
    print("Re-embedding external weights inline...")
    from onnx.external_data_helper import load_external_data_for_model
    load_external_data_for_model(onnx_model, os.path.dirname(onnx_path))
    onnx.save_model(onnx_model, onnx_path, save_as_external_data=False)
    onnx_model = onnx.load(onnx_path)

onnx.checker.check_model(onnx_model)
session = ort.InferenceSession(onnx_path)
test_obs = np.random.randn(4, OBS_SIZE).astype(np.float32)
with torch.no_grad():
    torch_out = wrapper(torch.from_numpy(test_obs)).numpy()
onnx_out = session.run(["actions"], {"obs": test_obs})[0]
max_diff = np.max(np.abs(torch_out - onnx_out))

print(f"Exported to: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)")
print(f"Max PyTorch vs ONNX diff: {max_diff:.2e}")
assert max_diff < 1e-5
print("Validation passed! (weights embedded inline)")
print(f"\nDrive path: hr-checkpoints-v2/{VERSION}/{VERSION}_jockey.onnx")
print(f"Copy to: apps/horse-racing/public/models/{VERSION}_jockey.onnx")